# Imputer

Ces transformers vont nous permettre de netoyer notre dataset en remplaçant ou en éliminant les données manquantes de notre dataset.

Pour ce faire nous alons avoir plusieurs transformers :
- SimpleImputer()
- IterativeImputer()
- MissingIndicator()
- KNNImputer()

# SimpleImputer()

Nous allons commencer par le SimpleImputer. Il permet de remplacer les valeurs manquantes par une valeur statistique.

## missing_values
Le premier paramètre à passer sera les missing_values, ici on pourra passer par exemple np.nan, qui sera assez fréquent, maintenant il peut arriver que les valeurs manquantes soient indiquée différement dans certains datasets, par exemple la valeurs -1 (qui en mémoire est représenter uniquement par des 1, donc 1111 1111 serait -1 sur une variable d'un byte de long). Par défaut missing_values est égal à np.nan.

## strategy
On aura plusieurs stratégies :
- mean : moyenne
- median : médiane
- most_frequent : la valeur la plus fréquente
- constant : une valeur constante

Par défaut la strategy est "mean".

## fill_value

uniquement pour la strategy constant, ce sera la valeur avec laquel remplacer la valeur NaN.


In [2]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline, make_union
from sklearn.preprocessing import StandardScaler, OneHotEncoder, Binarizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
import seaborn as sns

In [3]:
X = np.array([
                [12, 3],
                [2, 4],
                [5, 3],
                [np.nan, 3]
            ])

X_test = np.array(
    [
        [11, 5],
        [21, 2],
        [5, 10],
        [np.nan, np.nan]
    ]
)

In [4]:
imputer = SimpleImputer(missing_values=np.nan, strategy="mean")

imputer.fit_transform(X)

array([[12.        ,  3.        ],
       [ 2.        ,  4.        ],
       [ 5.        ,  3.        ],
       [ 6.33333333,  3.        ]])

In [5]:
imputer.transform(X_test)

array([[11.        ,  5.        ],
       [21.        ,  2.        ],
       [ 5.        , 10.        ],
       [ 6.33333333,  3.25      ]])

In [6]:
imputer = SimpleImputer(missing_values=np.nan, strategy="median")

imputer.fit_transform(X)

array([[12.,  3.],
       [ 2.,  4.],
       [ 5.,  3.],
       [ 5.,  3.]])

In [7]:
imputer.transform(X_test)

array([[11.,  5.],
       [21.,  2.],
       [ 5., 10.],
       [ 5.,  3.]])

On peut remarquer que sur le test set nos moyennes et mediane ne sont plus juste. C'est normal, notre transformer se base sur les données du train set.

Il faudra donc tendre à bien sélectionner notre train set et test set n'aient pas des moyennes médiane etc trop différentes l'un de l'autre.

Aussi on pourrais se dire qu'utiliser le test set et train set pour le calcul de la moyenne de notre imputer, mais celà provoquerais une fuite de données ce qui serait mauvais car notre estimator recevera, par ce biais, des informations sur le test set.

# KNNImputer

Le KNearest Neighbors Imputer est assez simple à comprendre, il va remplacer les valeurs manquantes d'un échantillon par les valeurs de l'échantillon qui lui ressemble le plus.

In [8]:
from sklearn.impute import KNNImputer

In [9]:
X = np.array([
                [12, 3, 5],
                [2, 4, 1],
                [5, 3, 9],
                [np.nan, 3, 5] 
            ])

X_test = np.array(
    [
        [11, 5, 1],
        [21, 2, 3],
        [5, np.nan, 15],
        [np.nan, np.nan, 5],
        [3, np.nan, 8]
    ]
)

In [10]:
imputer = KNNImputer(n_neighbors=1)
imputer.fit_transform(X)

array([[12.,  3.,  5.],
       [ 2.,  4.,  1.],
       [ 5.,  3.,  9.],
       [12.,  3.,  5.]])

In [11]:
imputer.transform(X_test)

array([[11.,  5.,  1.],
       [21.,  2.,  3.],
       [ 5.,  3., 15.],
       [12.,  3.,  5.],
       [ 3.,  3.,  8.]])

# Missing Indicator

Missing indicator va nous indiquer où il nous manque des valeurs dans notre dataset. Ce qui au premier abord peut parraitre inutile.

Mais celà s'avèrera utile lorsque l'on voudra par exemple trouver certains pattern dans notre dataset.

Par exemple, dans le cas du titanic, on pourrait tenter de trouver les personnes faisant partie de l'équipage en vérifiant ceux qui n'ont pas de donnée dans fare. Car il est normal que ceux-ci n'aient pas payé de ticket.

C'est pour ce genre de cas que le missing indicator pourra nous être utile.

In [12]:
from sklearn.impute import MissingIndicator

In [13]:
X = np.array(
    [
        [11, 5, 1],
        [21, 2, np.nan],
        [5, np.nan, 15],
        [np.nan, np.nan, 5] 
    ]
)

In [14]:
imputer = MissingIndicator()
imputer.fit_transform(X)

array([[False, False, False],
       [False, False,  True],
       [False,  True, False],
       [ True,  True, False]])

On va donc pouvoir créer des pipelines pour analyser le dataset et faire en sorte d'ajouter des informations utile à notre dataset via le make_union et le MissingIndicator.

In [15]:
X = np.array(
    [
        [11, 5, 1],
        [21, 2, 3],
        [5, np.nan, 15],
        [np.nan, np.nan, 5] 
    ]
)

pipeline = make_union(SimpleImputer(strategy='constant', fill_value=-1), MissingIndicator())

pipeline.fit_transform(X)

array([[11.,  5.,  1.,  0.,  0.],
       [21.,  2.,  3.,  0.,  0.],
       [ 5., -1., 15.,  0.,  1.],
       [-1., -1.,  5.,  1.,  1.]])

Ici nous avons donc un exemple du fait qu'un manque d'information peut être utiliser comme une information. Par exemple dans le titanic les decks les moins fréquent voir absent sont probablement ceux sur lesquels il y avait soit personne, soit ou l'entièreté des passagers sont morts.

In [16]:
data = sns.load_dataset('titanic')
X = data[['pclass', 'age']]
y = data['survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=5)

In [17]:
model = make_pipeline(KNNImputer(), SGDClassifier())

params = {
    'knnimputer__n_neighbors' : np.arange(1, 25)
}

grid = GridSearchCV(model, param_grid=params, cv=5)

grid.fit(X_train, y_train)

grid.best_params_

{'knnimputer__n_neighbors': 13}